# H-003 · Idiosyncratic Vol Rank

Factor test for **H-003** (equities): whether stocks with **lower** idiosyncratic volatility earn higher next-week returns (idiosyncratic volatility puzzle).

- **Idea** — Rank names by stock-specific return noise after stripping out market (SPY) exposure.
- **Claim** — Low idio-vol rank predicts higher forward returns; high rank predicts lower returns.
- **Why it might work** — Lottery preference and short-sale constraints can leave high idio-vol names overpriced; mispricing is easier to arbitrage in quieter names.
- **Data** — Daily OHLCV long panel + SPY daily returns (benchmark locked to SPY).

## What is idiosyncratic volatility (`idio_vol`)?

**Idiosyncratic volatility** is the volatility of a stock's returns *after removing market exposure*, not total realised vol.

1. Fit rolling OLS on daily log returns: `r_{i,t} = alpha + beta * r_{SPY,t} + epsilon_t`
2. Collect in-window residuals `epsilon_t`
3. **`idio_vol`** = sample std (`ddof=1`) of those residuals over the window (default 20 days)

This is **stock-specific noise** — moves not explained by the market — as opposed to beta-linked risk.

## Raw `idio_vol` vs the H-003 factor signal

| Object | What it is |
|--------|------------|
| **`idio_vol`** (raw) | Rolling residual std from the beta regression; an intermediate datum |
| **H-003 factor** | Cross-sectional percentile rank of `idio_vol` on each date |

The S1 factor panel notebook attaches **beta foundation columns only** (`alpha`, `beta`, `r2`). Idio vol is built here via modular helpers — not inline in the panel notebook — so research, walk-forward, and live code share the same implementation.

- Rolling OLS primitives: `data.processing.feature_implementation.beta`
- Idio vol series/panel helpers: `data.processing.feature_implementation.idiosyncratic_vol`
- Future store entrypoint (planned): `add_idiosyncratic_vol` in `feature_store` with `normalize=True` (CS pct-rank by default)

**Per-day residuals (`epsilon_t`):** `epsilon_t = r_{i,t} - alpha_t - beta_t * r_{SPY,t}`. Not stored in the panel by default; `idio_vol` summarizes their dispersion over the window. Per-day residuals could later support abnormal-return event studies, residual momentum, or conditional vol models.

## 0. Imports & Config

Resolve repo root; configure `window` (default 20), SPY benchmark dates, and `normalize` for the eventual store call. Import `fetch_ohlcv`, `fetch_top_n_equities`, `add_idiosyncratic_vol`, and `market_return_frame`.

## 1. Data Loading

Load daily OHLCV long panel (PIT universe) and SPY benchmark via project fetchers only.

- `fetch_top_n_equities` or saved `s1_factor_panel.parquet` for OHLCV (+ optional beta columns)
- `fetch_ohlcv("SPY", ...)` for market returns

## 2. Data Cleaning & Engineering

Build market log returns; attach raw `idio_vol` via `add_idiosyncratic_vol` (uses beta OLS under the hood).

Any winsorize / floor lives here — the library modules do **not** floor or winsorize by default.

## 3. Modeling / Signal Construction

Transform raw `idio_vol` into the H-003 factor: cross-sectional percentile rank within each date.

When implemented in `feature_store`, expect column `idio_vol_rank` (or similar mode name) with `normalize=True` by default.

## 4. Evaluation

Quintile spread and IC of idio-vol rank vs 5-day forward return; compare to total realised vol rank as baseline. Expect negative monotonicity (low rank → long). Use purge/embargo for overlapping 5d labels.